In [22]:
import torch
import json
import io
import numpy as np
import scipy.special as sp

import pickle as pkl
import zlib
import base64

In [23]:
import sys
sys.path.append('/home/kutulu/projects/code-of-kutulu-client')

In [24]:
from src.envs.agents.dqn_agent_ext import DQNAgentExt
from src.game.template import calculate_output_np

In [26]:
agents_dir = '../output/2025-05-26/20250526-171703'
agent_id = 0

In [33]:
checkpoint_dir = f'{agents_dir}/agent{agent_id}/2100'

In [34]:
with open(f'{agents_dir}/agents_info.json') as f:
    agents_info = json.load(f)
    info = agents_info[agent_id]
    del info['type']

In [35]:
agent = DQNAgentExt(**info)

In [36]:
agent.model.load_state_dict(torch.load(f"{checkpoint_dir}/model.pt"))

<All keys matched successfully>

In [37]:
agent.Train = False

In [38]:
data = {'entity_kind': [[1, 1, 3, 2, 0, 0, 0, 0, 0, 0]],
 'entity_features': [[[218.0, 3.0, 3.0, -3.0, 6.0, 0.0, 218.0],
   [221.0, 3.0, 1.0, -9.0, 10.0, 0.0, 221.0],
   [27.0, 0.0, 6.0, 6.0, 12.0, 0.0, 27.0],
   [2.0, -1.0, 7.0, 4.0, 11.0, 0.0, 2.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]],
 'entity_dir': [[[0.0, 6, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 12, 0.0],
   [0.0, 12, 0.0, 0.0, 0.0],
   [0.0, 15, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0]]]}

In [39]:
weights = {}
for k,v in agent.model.named_parameters():
    weights[k] = v.detach().numpy()
    print(k, v.shape)

kind_embs.weight torch.Size([13, 32])
features_linear.weight torch.Size([32, 7])
features_linear.bias torch.Size([32])
dir_linear.weight torch.Size([16, 5])
dir_linear.bias torch.Size([16])
entity_linear.weight torch.Size([16, 80])
entity_linear.bias torch.Size([16])
entity_impact.weight torch.Size([8, 80])
entity_impact.bias torch.Size([8])
out_linear.weight torch.Size([1, 16])
out_linear.bias torch.Size([1])


In [40]:
calculate_output_np(data, weights, num_classes=8)

array([[0.02323556, 0.0272663 , 0.02324413, 0.02391161, 0.02411188,
        0.02408639, 0.02323922, 0.34314003]])

In [41]:
tensor_data = {k: torch.tensor(v) for k,v in data.items()}

In [42]:
tensor_data = {
    'entity_kind': torch.IntTensor(data['entity_kind']),
    'entity_features': torch.FloatTensor(data['entity_features']),
    'entity_dir': torch.FloatTensor(data['entity_dir']),
}

In [43]:
model_output = agent.model(tensor_data)[0].detach().cpu().numpy()

In [44]:
model_output

array([0.02323553, 0.02726631, 0.02324408, 0.0239116 , 0.02411187,
       0.02408636, 0.02323917, 0.34314144], dtype=float32)

In [45]:
# data2, data1 = zip(*weights.items())

# data1 = pkl.dumps(data1)
# data2 = pkl.dumps(data2)

In [46]:
data1 = []
data2 = []
for k, v in weights.items():
    data2.append(k)
    buffer = io.BytesIO()
    np.save(buffer, v)
    data1.append(buffer.getvalue())

data1 = pkl.dumps(data1)
data2 = pkl.dumps(data2)

In [47]:
with open('../src/game/template.py') as f:
    lines = f.readlines()

In [48]:
with open('../src/game/template_submit.py', 'w') as f:
    for line in lines:
        line = line.replace("b'data1data1data1'", str(base64.b64encode(zlib.compress(data1, level=9))))
        line = line.replace("b'data2data2data2'", str(base64.b64encode(zlib.compress(data2, level=9))))
        line = line.replace("mode = 'mode'", "mode = 'dqn_ext'")
        line = line.replace("USED_ACTIONS = DEFAULT_KUTULU_ACTIONS", "USED_ACTIONS = EXTENDED_KUTULU_ACTIONS")
        f.write(line)

In [49]:
!ls -lh ../src/game/template_submit.py

-rw-rw-r-- 1 kutulu kutulu 29K May 26 19:56 ../src/game/template_submit.py
